In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "pubmed_20k_rct"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())
print("Files:", list(DATA_DIR.glob("*.txt")))

from src.data.data_loader import load_pubmed_20k_rct

data = load_pubmed_20k_rct(DATA_DIR)
train_df, dev_df, test_df = data["train"], data["dev"], data["test"]

train_df.head()



PROJECT_ROOT: c:\Users\mansour\Documents\Clinical text classification
DATA_DIR exists: True
Files: [WindowsPath('c:/Users/mansour/Documents/Clinical text classification/data/raw/pubmed_20k_rct/dev.txt'), WindowsPath('c:/Users/mansour/Documents/Clinical text classification/data/raw/pubmed_20k_rct/test.txt'), WindowsPath('c:/Users/mansour/Documents/Clinical text classification/data/raw/pubmed_20k_rct/train.txt')]


,label,text
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...
1,METHODS,A total of 125 patients with primary knee OA w...
2,METHODS,Outcome measures included pain reduction and i...
3,METHODS,Pain was assessed using the visual analog pain...
4,METHODS,Secondary outcome measures included the Wester...


In [11]:
!pip install -q transformers datasets accelerate torch scikit-learn pandas


In [7]:
labels = sorted(train_df["label"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

print(labels)
print(label2id)


['BACKGROUND', 'CONCLUSIONS', 'METHODS', 'OBJECTIVE', 'RESULTS']
{'BACKGROUND': 0, 'CONCLUSIONS': 1, 'METHODS': 2, 'OBJECTIVE': 3, 'RESULTS': 4}


In [12]:
from datasets import Dataset, DatasetDict

train_hf = Dataset.from_pandas(train_df[["text","label"]].copy())
dev_hf   = Dataset.from_pandas(dev_df[["text","label"]].copy())
test_hf  = Dataset.from_pandas(test_df[["text","label"]].copy())

raw_datasets = DatasetDict({"train": train_hf, "dev": dev_hf, "test": test_hf})
raw_datasets


c:\Users\mansour\miniconda3\envs\dp4ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 180040
    })
    dev: Dataset({
        features: ['text', 'label'],
        num_rows: 30212
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 30135
    })
})

In [13]:
def encode_labels(example):
    return {"labels": label2id[example["label"]]}

raw_datasets = raw_datasets.map(encode_labels)
raw_datasets = raw_datasets.remove_columns(["label"])
raw_datasets["train"].column_names


Map: 100%|██████████| 30135/30135 [00:00<00:00, 42993.52 examples/s]


['text', 'labels']

In [16]:
def encode_labels(example):
    return {"labels": label2id[example["label"]]}

raw_datasets = raw_datasets.map(encode_labels)
raw_datasets = raw_datasets.remove_columns(["label"])
raw_datasets["train"].column_names


Map:   0%|          | 0/180040 [00:00<?, ? examples/s]


KeyError: 3

In [17]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)
tokenized_datasets


c:\Users\mansour\miniconda3\envs\dp4ai\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mansour\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 30135/30135 [00:03<00:00, 8593.38 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 180040
    })
    dev: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30212
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30135
    })
})

In [18]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 479.08it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


In [22]:
from transformers import TrainingArguments

OUT_DIR = PROJECT_ROOT / "artifacts" / "distilbert"

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,

    # IMPORTANT: use eval_strategy (newer) if evaluation_strategy fails
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    logging_steps=100,
    report_to="none",
)


In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


NameError: name 'model' is not defined

In [ ]:
trainer.train()


In [24]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

pred = trainer.predict(tokenized_datasets["dev"])
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=1)

# convert ids → label strings for readability
y_true_lbl = [id2label[i] for i in y_true]
y_pred_lbl = [id2label[i] for i in y_pred]

report = classification_report(
    y_true_lbl, y_pred_lbl, labels=labels, output_dict=True, zero_division=0
)
cm = confusion_matrix(y_true_lbl, y_pred_lbl, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print("Dev accuracy:", report["accuracy"])
pd.DataFrame(report).T.head(10), cm_df


NameError: name 'trainer' is not defined

In [ ]:
import json

OUT_DIR.mkdir(parents=True, exist_ok=True)

with (OUT_DIR / "metrics_dev.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

cm_df.to_csv(OUT_DIR / "confusion_matrix_dev.csv", index=True)

print("Saved:")
print("-", OUT_DIR / "metrics_dev.json")
print("-", OUT_DIR / "confusion_matrix_dev.csv")
